# dHS gene selection
This script will use ClinGen data to generate a list of putative dominant and haplosufficient genes

### Load libraries

In [1]:
import json
from math import isnan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import urllib.request

### Set data path variables

In [ ]:
# data filepaths
gene_disease_fp = '../../data/clingen/Clingen-Gene-Disease-Summary-2025-11-13.csv'
dosage_sensitivity_fp = '../../data/clingen/Clingen-Dosage-Sensitivity-2025-11-13.csv'
savedir='../../data/dnd_hgnc'
inheritance_mode_filters=['Definitive', 'Strong', 'Moderate']
dominance_filters_keep = ['AD','SD'] # autosomal dominant and semidominant conditions
dominance_filters_remove = ['XL'] # remove x-linked conditions
haploinsufficiency_levels = ['SufficientEvidence', 'EmergingEvidence'] #'LittleEvidence'] # annotations to consider a gene 'haploinsufficient'

### Load data

In [3]:
# load modes of inheritance data
gd = pd.read_csv(gene_disease_fp)
cols = gd.iloc[3].values
gd = gd[5:]
gd.columns = cols

# load dosage sensitivity data
ds = pd.read_csv(dosage_sensitivity_fp)
ds[['GeneSymbol','HGNC_ID']] = ds['Gene Symbol /Region Name'].str.split('HGNC', expand=True)
ds['HGNC/Dosage ID'] = 'HGNC'+ds['HGNC_ID']
ds.rename(columns={'HGNC/Dosage ID':'GENE ID (HGNC)'},inplace=True)
ds.drop(labels=['HGNC_ID'], inplace=True, axis=1)

In [36]:
gd

,GENE SYMBOL,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,SOP,CLASSIFICATION,ONLINE REPORT,CLASSIFICATION DATE,GCEP
5,AARS1,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,SOP10,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2024-03-14T16:00:00.000Z,Charcot-Marie-Tooth Disease Gene Curation Expe...
6,AARS2,HGNC:21022,mitochondrial disease,MONDO:0044970,AR,SOP8,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2022-04-18T16:00:00.000Z,Mitochondrial Diseases Gene Curation Expert Panel
7,ABAT,HGNC:23,developmental and epileptic encephalopathy,MONDO:0100062,AR,SOP8,Moderate,https://search.clinicalgenome.org/kb/gene-vali...,2022-04-19T16:00:00.000Z,Epilepsy Gene Curation Expert Panel
8,ABCA3,HGNC:33,interstitial lung disease due to ABCA3 deficiency,MONDO:0012582,AR,SOP10,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2024-09-17T16:00:00.000Z,Interstitial Lung Disease Gene Curation Expert...
9,ABCA4,HGNC:34,ABCA4-related retinopathy,MONDO:0800406,AR,SOP9,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2022-10-06T16:00:00.000Z,Retina Gene Curation Expert Panel
...,...,...,...,...,...,...,...,...,...,...
3331,NIPBL,HGNC:28862,Cornelia de Lange syndrome,MONDO:0016033,AD,SOP11,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2025-08-11T10:00:00.000Z,Intellectual Disability and Autism Gene Curati...
3332,CP,HGNC:2295,aceruloplasminemia,MONDO:0011426,AR,SOP11,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2025-10-27T04:00:00.000Z,General Inborn Errors of Metabolism Gene Curat...
3333,POFUT1,HGNC:14988,Dowling-Degos disease 2,MONDO:0014130,AD,SOP11,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2025-10-20T17:00:00.000Z,Congenital Disorders of Glycosylation Gene Cur...
3334,RERE,HGNC:9965,complex neurodevelopmental disorder with or wi...,MONDO:0100465,AD,SOP9,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2023-03-21T11:00:00.000Z,Intellectual Disability and Autism Gene Curati...


In [4]:
print('Total number of genes in gene-disease table: ' + str(len(gd['GENE ID (HGNC)'].unique())))

Total number of genes in gene-disease table: 2753


In [5]:
print('Total number of diseases in gene-disease table: ' + str(len(gd['DISEASE LABEL'].unique())))

Total number of diseases in gene-disease table: 1912


In [6]:
ds['HI Score'].unique()

array(['AutosomalRecessive', 'SufficientEvidence', 'NoEvidence',
       'LittleEvidence', 'SensitivityUnlikely', 'EmergingEvidence', nan],
      dtype=object)

### Filter for genes linked to dominant conditions

In [7]:
# filter the modes of inheritance to only include those in our specified filters
hc = gd[gd['CLASSIFICATION'].isin(inheritance_mode_filters)] # hc for 'high-confidence'
ad_hc = hc[hc['MOI'].isin(dominance_filters_keep)]
ad_hc_genelist=list(ad_hc['GENE SYMBOL'].unique())
print(len(ad_hc_genelist))

# get the genes with X-linked inheritance
xl = gd[gd['MOI']=='XL'] # note that we've elected to not remove recessive genes, since a gene can exhibit both dominant and recessive conditions
print(len(xl['GENE SYMBOL'].unique()))
xl_genelist = list(xl['GENE SYMBOL'].unique())

# now filter out any genes that are in the xl_genelist from the ad_hc gene list
ad_hc_notXL_genes = [x for x in ad_hc_genelist if x not in xl_genelist]
print(len(set(ad_hc_notXL_genes)))

# final filtering to get the gene set information
ad_hc_notXL = ad_hc[ad_hc['GENE SYMBOL'].isin(ad_hc_notXL_genes)]

855
179
855


### Filter out genes that have haploINsufficiency annotations

In [8]:
# filter out haploinsufficient genes using dosage sensitivity data
merged = ad_hc_notXL.merge(ds, how='left', on=['GENE ID (HGNC)']) # merge dominant genes with dosage data
ad_hc_notXL_HI = merged[merged['HI Score'].isin(haploinsufficiency_levels)]
ad_hc_notXL_HS = merged[~merged['HI Score'].isin(haploinsufficiency_levels)]
print('Dominant and Haplosufficient (by lack of haploinsuffiency annotation): ' + str(len(ad_hc_notXL_HS['GENE SYMBOL'].unique())))
print('Dominant and Haploinsufficient: ' + str(len(ad_hc_notXL_HI['GENE SYMBOL'].unique())))
print('Total dominant (and semidominant & not X-linked): ' + str(len(ad_hc_notXL['GENE SYMBOL'].unique())))

Dominant and Haplosufficient (by lack of haploinsuffiency annotation): 598
Dominant and Haploinsufficient: 257
Total dominant (and semidominant & not X-linked): 855


### Checking positive and negative controls

In [9]:
'HNF4A' in list(ad_hc_notXL_HS['GENE SYMBOL'])

False

In [10]:
'GCK' in list(ad_hc_notXL_HS['GENE SYMBOL'])

False

### Save the resulting data frame

In [ ]:
# save the dHS set
ad_hc_notXL_HS.to_csv(savedir + '/dnd_set.csv')

In [26]:
### Check a few positive controls
print('NEFL' in list(ad_hc_notXL_HS['GENE SYMBOL']))
print('MFN2' in list(ad_hc_notXL_HS['GENE SYMBOL']))
print('FUS' in list(ad_hc_notXL_HS['GENE SYMBOL']))
print('MYH7' in list(ad_hc_notXL_HS['GENE SYMBOL']))

True
True
True
True


In [27]:
for gene in list(ad_hc_notXL_HS['GENE SYMBOL']):
    print(gene)

AARS1
ABCB6
ABCC8
ACD
ACO2
ACOX1
ACTA1
ACTB
ACTC1
ACTG1
ACTG1
ACTL6A
ACTN1
ACVR1
ADAR
AFG3L2
AGO2
AKT3
ALDH18A1
ALG9
ANK1
ANKRD26
ANXA11
AP1G1
AP2M1
ARF1
ASXL2
ATL1
ATL3
ATP13A3
ATP1A1
ATP1A3
ATXN2
BACH2
BEST1
BRD4
BRSK2
BSCL2
C19orf12
C1QTNF5
C3
C3
C9orf72
CACNA1C
CACNA1E
CACNA1S
CALM1
CALM1
CALM2
CALM2
CALM3
CALM3
CAPN5
CARD11
CARD11
CASQ2
CASR
CASR
CAV3
CBL
CD46
CDC42
CDH11
CDK4
CEBPA
CEL
CFB
CFH
CFI
CFI
CHAMP1
CHCHD10
CHCHD10
CHD3
CHD4
CHMP2B
CHN1
CHRNA4
CHRNB2
CLCN7
CLCN7
CNNM2
CNOT3
COCH
COL10A1
COL4A3
COL4A4
COL5A2
COL6A1
COL6A2
COL6A3
CPOX
CRX
CSNK2A1
CSNK2B
CSRP3
CTNND1
CUX2
CXCR4
DCC
DCTN1
DDR2
DES
DIAPH1
DNM1L
DNM2
DNM2
DPF2
DSC2
DSPP
DYNC1H1
EEF1A2
EFEMP1
EGFR
EGR2
EIF2S3
ELANE
ELP4
EMC1
EPCAM
EPHB4
ETS1
EZH2
F11
F2
F5
FAM111B
FBLN5
FBXO38
FGA
FGB
FGF12
FGF8
FGFR2
FGFR2
FGFR2
FGFR2
FGFR3
FGFR3
FGG
FHOD3
FLT4
FN1
FN1
FOXN1
FUS
FXYD2
GABBR2
GABRB3
GABRD
GABRG2
GANAB
GARS1
GBA1
GDAP1
GDF2
GDF2
GFAP
GFI1B
GJB6
GLUD1
GNAI1
GNAO1
GNAT1
GNB4
GP1BA
GREM1
GRIA1
GRIA2
GRIA4
GRIK2
GRI

In [14]:
print(len(ad_hc_notXL_HS['GENE SYMBOL'].unique()))

598


In [25]:
ad_hc_notXL_HS[ad_hc_notXL_HS['HI Score'].isin(['SensitivityUnlikely'])]

,GENE SYMBOL,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,SOP,CLASSIFICATION,ONLINE REPORT,CLASSIFICATION DATE,GCEP,...,Gene Symbol /Region Name,GRCh37,HI Score,TS Score,OMIMMorbid,%HI,pLI,LOEUF,LastEvaluated Date,GeneSymbol
402,PCSK9,HGNC:20001,"hypercholesterolemia, autosomal dominant, 3",MONDO:0011369,AD,SOP7,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2018-11-14T17:00:00.000Z,General Gene Curation Expert Panel,...,PCSK9HGNC:20001,1:55505221-55530525,SensitivityUnlikely,NoEvidence,OMIM Morbid,73.06,0,1.25,12/17/2015,PCSK9


In [ ]:
s_het = pd.read_csv('../../data/s_het/s_het_estimates.genebayes.tsv',sep='\t')
s_het[['identifier','hgnc_symbol']] = s_het["hgnc"].str.split(":", expand=True)
s_het

,ensg,hgnc,chrom,obs_lof,exp_lof,prior_mean,post_mean,post_lower_95,post_upper_95,identifier,hgnc_symbol
0,ENSG00000198488,HGNC:24141,chr11,12.0,8.9777,0.001997,0.000065,0.000007,0.000178,HGNC,24141
1,ENSG00000164363,HGNC:26441,chr5,31.0,28.5500,0.000973,0.000161,0.000026,0.000442,HGNC,26441
2,ENSG00000159337,HGNC:30038,chr15,28.0,41.8400,0.000856,0.000170,0.000019,0.000533,HGNC,30038
3,ENSG00000177558,HGNC:26366,chr19,12.0,9.9101,0.001040,0.000175,0.000025,0.000487,HGNC,26366
4,ENSG00000062524,HGNC:6721,chr15,40.0,42.6820,0.002000,0.000180,0.000012,0.000589,HGNC,6721
...,...,...,...,...,...,...,...,...,...,...,...
19066,ENSG00000108298,HGNC:10312,chr17,0.0,12.2410,0.349655,0.777394,0.188250,0.999947,HGNC,10312
19067,ENSG00000142937,HGNC:10441,chr1,0.0,13.8460,0.367146,0.781929,0.176357,0.999976,HGNC,10441
19068,ENSG00000231500,HGNC:10401,chr6,0.0,11.4500,0.407376,0.823303,0.190498,0.999999,HGNC,10401
19069,ENSG00000108055,HGNC:2468,chr10,0.0,79.4900,0.376332,0.841343,0.460300,0.998543,HGNC,2468


In [31]:
s_het[s_het['hgnc']=='HGNC:14078']

,ensg,hgnc,chrom,obs_lof,exp_lof,prior_mean,post_mean,post_lower_95,post_upper_95,identifier,hgnc_symbol
18255,ENSG00000112182,HGNC:14078,chr6,2.0,27.02,0.280046,0.246151,0.12572,0.406918,HGNC,14078


In [22]:
print(notAR_HI['HI Score'].unique())
notAR_HI

['NoEvidence' nan 'SensitivityUnlikely']


,GENE SYMBOL,GENE ID (HGNC),DISEASE LABEL,DISEASE ID (MONDO),MOI,SOP,CLASSIFICATION,ONLINE REPORT,CLASSIFICATION DATE,GCEP,...,Gene Symbol /Region Name,GRCh37,HI Score,TS Score,OMIMMorbid,%HI,pLI,LOEUF,LastEvaluated Date,GeneSymbol
0,AARS1,HGNC:20,Charcot-Marie-Tooth disease axonal type 2N,MONDO:0013212,AD,SOP10,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2024-03-14T16:00:00.000Z,Charcot-Marie-Tooth Disease Gene Curation Expe...,...,AARS1HGNC:20,16:70286201-70323409,NoEvidence,NoEvidence,OMIM Morbid,24.74,0,0.77,01/11/2018,AARS1
1,ABCB6,HGNC:47,dyschromatosis universalis hereditaria 3,MONDO:0014169,AD,SOP10,Moderate,https://search.clinicalgenome.org/kb/gene-vali...,2024-04-26T18:00:00.000Z,General Inborn Errors of Metabolism Gene Curat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ACD,HGNC:25070,ACD-related short telomere syndrome,MONDO:0100569,SD,SOP10,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2024-02-20T17:00:00.000Z,Interstitial Lung Disease Gene Curation Expert...,...,ACDHGNC:25070,16:67691415-67694163,NoEvidence,NoEvidence,OMIM Morbid,81.12,0,0.92,04/24/2025,ACD
4,ACO2,HGNC:118,mitochondrial disease,MONDO:0044970,AD,SOP9,Strong,https://search.clinicalgenome.org/kb/gene-vali...,2023-10-19T04:00:00.000Z,Mitochondrial Diseases Gene Curation Expert Panel,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ACTA1,HGNC:129,alpha-actinopathy,MONDO:0100084,AD,SOP10,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2024-02-26T17:00:00.000Z,Congenital Myopathies Gene Curation Expert Panel,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,CHRNB1,HGNC:1961,congenital myasthenic syndrome 2A,MONDO:0014581,AD,SOP11,Moderate,https://search.clinicalgenome.org/kb/gene-vali...,2025-10-27T16:00:00.000Z,Congenital Myopathies Gene Curation Expert Panel,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
964,PPP2R1A,HGNC:9302,complex neurodevelopmental disorder,MONDO:0100038,AD,SOP9,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2022-11-16T11:00:00.000Z,Intellectual Disability and Autism Gene Curati...,...,PPP2R1AHGNC:9302,19:52693305-52732771,NoEvidence,NoEvidence,OMIM Morbid,23.46,1,0.35,12/06/2023,PPP2R1A
965,PPP2CA,HGNC:9299,complex neurodevelopmental disorder,MONDO:0100038,AD,SOP11,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2025-05-07T19:00:00.000Z,Intellectual Disability and Autism Gene Curati...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967,POFUT1,HGNC:14988,Dowling-Degos disease 2,MONDO:0014130,AD,SOP11,Definitive,https://search.clinicalgenome.org/kb/gene-vali...,2025-10-20T17:00:00.000Z,Congenital Disorders of Glycosylation Gene Cur...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Optional: gather important alias naming information on these genes from the HGNC biomart page: https://www.genenames.org/tools/multi-symbol-checker/
* Click 'Gene' and input list of dominant and haplosufficient gene names
* This essentially standardizes the gene names to their approved HGNC symbol
* Save it to data/dnd_hgnc/dnd_hgnc_mapped.csv

### View mapped and standardized gene information

In [ ]:
# load mapped & standardized gene names from HGNC
standardized = pd.read_csv(savedir+'/dnd_hgnc_mapped.csv',comment='s')
print(len(standardized.Input.unique()))
standardized


KeyboardInterrupt

